In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :powerlaw

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model] Fitting chain 3 (tau=59)
[ Info: [powerlaw] iter 1000/1000000 elapsed=5.7s, rate=0.136, mean=[0.706, 0.00179, 0.636, 0.781], std=[0.0498, 0.000489, 0.0821, 0.0213] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=10.4s, rate=0.119, mean=[0.732, 0.00156, 0.634, 0.757], std=[0.0426, 0.000411, 0.0587, 0.0265] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=14.2s, rate=0.110, mean=[0.750, 0.00143, 0.624, 0.750], std=[0.0430, 0.000377, 0.0496, 0.0242] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=18.1s, rate=0.104, mean=[0.759, 0.00135, 0.614, 0.746], std=[0.0411, 0.000352, 0.0457, 0.0230] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=21.9s, rate=0.098, mean=[0.760, 0.00130, 0.607, 0.750], std=[0.0379, 0.000331, 0.0434, 0.0219] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=25.8s, rate=0.097, mean=[0.758, 0.00126, 0.598, 0.751], std=[0.0350, 0.000313, 0.0436, 0.0204] [ADAPT]
[ Info: [powerlaw] iter 7000/1000000 elapsed=29.6s, rate=0.094,